In [1]:
import sys
import os
sys.path.insert(0,os.path.abspath('..'))

from glob import glob
from tqdm import tqdm
from onnx2torch import convert
import matplotlib.pyplot as plt
from src.utilities.get_device import Device
from src.utilities.load_dataset import load_dataset
from skl2onnx.helpers.onnx_helper import load_onnx_model
from src.utilities.evaluate_model import evaluate_validation
from src.utilities.dataloader_generator import generate_dataloader

In [2]:
dataloaders_dict = dict(
    train=generate_dataloader(load_dataset(), batch_size=32),
    validation=generate_dataloader(load_dataset(
        folder="test"), batch_size=32)
)
device = Device().get_device()

In [3]:
directory = "../_results_without_finetune/evaluation_before_finetuning"
if not os.path.exists(directory):
    os.makedirs(directory)

In [4]:
models_location = sorted(glob('../_results_without_finetune/1713121759_result_BS_32_MD_16_T_0_TT_0.5_K_5/*.onnx'))
max_accuracy = 0
model_max_accuary = ""
for i, model_location in tqdm(enumerate(models_location), position=0, leave=True):
    model_name = model_location.split("/")[-1]
    onnx_model = load_onnx_model(model_location)
    torch_model = convert(onnx_model)
    torch_model.to(device)
    accuracy = evaluate_validation(device, torch_model, dataloaders_dict['validation'])
    accuarcy = accuracy.item()
    if accuracy > max_accuracy:
        max_accuracy = accuarcy
        model_max_accuary = model_name
print(max_accuracy, model_max_accuary)

53it [04:37,  5.23s/it]

0.8867413993042134 net015.onnx
